# Kairos Reality Check — Load Weights & Test on Real Scenarios

Loads pre-trained CIDA + PlanProjector weights from Google Drive
and tests Kairos on **real-world scenarios** before public launch.

**Runtime**: T4 or A100 GPU.

In [ ]:
!pip install -q torch transformers accelerate torchdiffeq
import subprocess, sys, os
repo = 'CIDA-plugin'
if not os.path.exists(repo):
    subprocess.run(['git','clone','--depth','1',
                   'https://github.com/Kairatzh/CIDA-plugin.git'], check=True)
sys.path.insert(0, os.path.abspath(repo))
from google.colab import drive
drive.mount('/content/drive')
print("Setup complete.")


## 1. Load Frozen LLM + CIDA Weights

In [ ]:
import gc, re, json, time, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from cida_plugin import CIDAPlugin, CIDAPluginConfig

DEVICE = "cuda"
WEIGHTS_PATH = '/content/drive/MyDrive/Kairos_CIDA_7B/best'

def flush():
    gc.collect()
    torch.cuda.empty_cache()

info_path = Path(WEIGHTS_PATH) / "train_info.json"
kairos_cfg_path = Path(WEIGHTS_PATH) / "kairos_config.json"
if info_path.exists():
    with open(info_path) as f: info = json.load(f)
elif kairos_cfg_path.exists():
    with open(kairos_cfg_path) as f: info = json.load(f)
else:
    info = {}

MODEL_NAME = info.get("llm_model_name", "Qwen/Qwen1.5-7B-Chat")
NUM_PLAN_TOKENS = info.get("num_plan_tokens", 8)
print(f"LLM: {MODEL_NAME}, Plan tokens: {NUM_PLAN_TOKENS}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True,
    torch_dtype=torch.float16, device_map="auto",
    output_hidden_states=True,
)
for p in llm.parameters(): p.requires_grad = False
llm.eval()

LLM_HIDDEN = llm.config.hidden_size
embed_tokens = llm.get_input_embeddings()
for p in embed_tokens.parameters(): p.requires_grad = False
flush()
print(f"LLM loaded: {sum(p.numel() for p in llm.parameters())/1e9:.1f}B params (frozen)")


In [ ]:
cida_cfg = CIDAPluginConfig.from_pretrained(WEIGHTS_PATH)
cida = CIDAPlugin(cida_cfg)
sd = torch.load(str(Path(WEIGHTS_PATH) / "pytorch_model.bin"), map_location="cpu")
cida.load_state_dict(sd)
cida = cida.to(DEVICE).float().eval()
del sd; flush()

class CIDAPlanProjector(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=8):
        super().__init__()
        self.num_plan_tokens = num_plan_tokens
        self.d_llm = d_llm
        d_flat = num_agents * d_cida
        self.projector = nn.Sequential(
            nn.Linear(d_flat, d_flat * 2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d_flat * 2, num_plan_tokens * d_llm),
            nn.LayerNorm(num_plan_tokens * d_llm),
        )
    def forward(self, state_s):
        B = state_s.size(0)
        flat = state_s.reshape(B, -1).float()
        return self.projector(flat).view(B, self.num_plan_tokens, self.d_llm)

plan_projector = CIDAPlanProjector(
    d_cida=cida_cfg.d_hidden, d_llm=LLM_HIDDEN,
    num_agents=cida_cfg.num_agents, num_plan_tokens=NUM_PLAN_TOKENS,
).to(DEVICE)
plan_projector.load_state_dict(
    torch.load(str(Path(WEIGHTS_PATH) / "plan_projector.pt"), map_location=DEVICE)
)
plan_projector.eval()
flush()

total_p = sum(p.numel() for p in cida.parameters()) + sum(p.numel() for p in plan_projector.parameters())
print(f"CIDA + Projector: {total_p/1e6:.2f}M params loaded from Drive")


## 2. Generation Helpers (with OOM protection)

In [ ]:
@torch.no_grad()
def get_context(input_ids, attention_mask):
    out = llm(input_ids=input_ids, attention_mask=attention_mask,
              output_hidden_states=True, use_cache=False)
    last_h = out.hidden_states[-1]
    seq_len = attention_mask.sum(dim=1) - 1
    bidx = torch.arange(input_ids.size(0), device=DEVICE)
    pooled = last_h[bidx, seq_len].float()
    p_emb = embed_tokens(input_ids).detach().float()
    del out, last_h
    flush()
    return pooled, p_emb

@torch.no_grad()
def generate_base(prompt, max_new=64):
    flush()
    tokenizer.padding_side = "left"
    enc = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(DEVICE)
    t0 = time.time()
    ids = llm.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                       max_new_tokens=max_new, do_sample=False,
                       pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    plen = enc["input_ids"].size(1)
    text = tokenizer.decode(ids[0, plen:], skip_special_tokens=True)
    ntok = ids.size(1) - plen
    del enc, ids; flush()
    return text, elapsed, ntok

@torch.no_grad()
def generate_kairos(prompt, max_new=64):
    flush()
    tokenizer.padding_side = "left"
    enc = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(DEVICE)
    pooled, p_emb = get_context(enc["input_ids"], enc["attention_mask"])
    t0 = time.time()
    cida_out = cida(pooled=pooled)
    plan_emb = plan_projector(cida_out["state_final"].s)
    cat_emb = torch.cat([p_emb, plan_emb], dim=1).to(next(llm.parameters()).dtype)
    del p_emb, plan_emb, pooled; flush()
    pmask = torch.ones(1, NUM_PLAN_TOKENS, dtype=enc["attention_mask"].dtype, device=DEVICE)
    gmask = torch.cat([enc["attention_mask"], pmask], dim=1)
    ids = llm.generate(inputs_embeds=cat_emb, attention_mask=gmask,
                       max_new_tokens=max_new, do_sample=False,
                       pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    text = tokenizer.decode(ids[0], skip_special_tokens=True)
    disag = cida_out["disagreement"].mean().item()
    del enc, ids, cat_emb, gmask, cida_out; flush()
    return text, elapsed, disag

def make_prompt(q, task="math"):
    if task == "math":
        s = "You are a mathematical reasoner. Solve step by step, then state the final answer after '####'."
    elif task == "logic":
        s = "You are a logical reasoner. Think step by step and give the correct answer."
    elif task == "code":
        s = "You are a Python programming expert. Write clean, correct code."
    else:
        s = "You are a helpful assistant."
    return f"<|im_start|>system\n{s}\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"

print("Helpers ready (with OOM protection).")


## 3. Reality Check: 12 Real-World Scenarios

6 categories: Easy Math, Hard Math, Logic, Code, Creative, Noisy Input

In [ ]:
SCENARIOS = [
    {"cat": "Easy Math", "task": "math",
     "q": "A shop sells apples for $2 each. Tom buys 5 apples and pays with a $20 bill. How much change does he get?",
     "gold": "10"},
    {"cat": "Easy Math", "task": "math",
     "q": "Lisa has 24 cookies. She gives 1/3 to her friend and eats 4 herself. How many cookies does she have left?",
     "gold": "12"},
    {"cat": "Hard Math", "task": "math",
     "q": "A train travels at 60 mph for 2.5 hours, then at 80 mph for 1.5 hours. What is the total distance?",
     "gold": "270"},
    {"cat": "Hard Math", "task": "math",
     "q": "An item costs $80. A 25% sale discount is applied, then an additional 10% loyalty discount on the sale price. What is the final price?",
     "gold": "54"},
    {"cat": "Logic", "task": "logic",
     "q": "All roses are flowers. All flowers need water. Do roses need water? Answer Yes or No.",
     "gold": "Yes"},
    {"cat": "Logic", "task": "logic",
     "q": "In a race, Alex finished before Ben. Ben finished before Carlos. Did Alex finish before Carlos? Answer Yes or No.",
     "gold": "Yes"},
    {"cat": "Code", "task": "code",
     "q": "Write a Python function called 'fibonacci' that takes integer n and returns the n-th Fibonacci number using iteration.",
     "gold": "def fibonacci"},
    {"cat": "Code", "task": "code",
     "q": "Write a Python function called 'is_palindrome' that checks if a string is a palindrome. Return True or False.",
     "gold": "def is_palindrome"},
    {"cat": "Creative", "task": "chat",
     "q": "Write a short friendly email congratulating a colleague on a promotion.",
     "gold": "congratulat"},
    {"cat": "Creative", "task": "chat",
     "q": "Describe a sunset over the ocean in 2-3 sentences.",
     "gold": "sun"},
    {"cat": "Noisy", "task": "math",
     "q": "Janett slles 16 egsg a dya. Sh eats 3 fr brekfast. Hw mny can sh sll per dya?",
     "gold": "13"},
    {"cat": "Noisy", "task": "logic",
     "q": "Al rsoes ar flwers. Al flwers ned watr. Do rses ned watr? Answr Ys or N.",
     "gold": "Yes"},
]

print("=" * 85)
print(f"  {'#':<3} {'Category':<12} {'Model':<8} {'Time':>6}  {'Status':<6}  Answer Preview")
print("=" * 85)

results = {}
for i, sc in enumerate(SCENARIOS):
    prompt = make_prompt(sc["q"], sc["task"])
    gold = sc["gold"].lower().strip()
    cat = sc["cat"]

    base_text, base_t, _ = generate_base(prompt, max_new=64)
    base_hit = gold in base_text.lower()[:200]

    kairos_text, kairos_t, disag = generate_kairos(prompt, max_new=64)
    kairos_hit = gold in kairos_text.lower()[:200]

    bs = "PASS" if base_hit else "FAIL"
    ks = "PASS" if kairos_hit else "FAIL"

    print(f"  {i+1:<3} {cat:<12} {'Base':<8} {base_t:>5.1f}s  {bs:<6}  {base_text.strip()[:55]}")
    print(f"      {'':<12} {'Kairos':<8} {kairos_t:>5.1f}s  {ks:<6}  {kairos_text.strip()[:55]}")
    print(f"      {'':<12} disag={disag:.3f}  gold='{sc['gold']}'")
    print("-" * 85)

    if cat not in results:
        results[cat] = {"bp": 0, "kp": 0, "n": 0}
    results[cat]["n"] += 1
    if base_hit: results[cat]["bp"] += 1
    if kairos_hit: results[cat]["kp"] += 1

print()
print("=" * 65)
print(f"  {'Category':<15} {'N':>3}  {'Base':>8}  {'Kairos':>8}  {'Verdict'}")
print("=" * 65)
tb, tk, tn = 0, 0, 0
for cat, r in results.items():
    v = "KAIROS" if r['kp'] > r['bp'] else ("TIE" if r['kp'] == r['bp'] else "BASE")
    print(f"  {cat:<15} {r['n']:>3}  {r['bp']}/{r['n']:>6}  {r['kp']}/{r['n']:>6}  {v}")
    tb += r['bp']; tk += r['kp']; tn += r['n']

print("=" * 65)
ready = "READY" if tk >= tn * 0.6 else "NEEDS WORK"
print(f"\n  Kairos pass rate: {tk}/{tn} ({tk/tn*100:.0f}%)")
print(f"  Base  pass rate: {tb}/{tn} ({tb/tn*100:.0f}%)")
print(f"  Launch status  : {ready}")


## 4. Side-by-Side Deep Dive

In [ ]:
IDX = 0  # change to inspect any scenario

sc = SCENARIOS[IDX]
prompt = make_prompt(sc["q"], sc["task"])

print(f"Category : {sc['cat']}")
print(f"Question : {sc['q']}")
print(f"Gold     : {sc['gold']}\n")

base_text, base_t, _ = generate_base(prompt, max_new=100)
kairos_text, kairos_t, disag = generate_kairos(prompt, max_new=100)

print("=" * 60)
print("  BASE LLM:")
print("=" * 60)
print(base_text[:400])
print()
print("=" * 60)
print("  KAIROS:")
print("=" * 60)
print(kairos_text[:400])
print(f"\n  Disagreement: {disag:.4f}")
